# Random forests & boosting

_Machine Learning_

**Combine many weak models into one strong one.**

One tree is weak and easily fooled by noise.

     An ensemble combines many models and lets them vote.

---

This notebook compares a single decision tree against two ensembles — a **random forest** and **gradient boosting** — one step at a time, first on synthetic data and then on real breast-cancer scans. Run each cell and read the note above it to see why combining many trees wins. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Make a dataset and split it for honest scoring

We generate 600 synthetic examples with 10 features (5 of them actually informative). We then hold out 25% of the data as a **test set** the models never see during training, so the accuracy we report later reflects how well each model generalizes rather than memorizes.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=600, n_features=10, n_informative=5,
                           random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

### Step 2 — Train one tree and two ensembles

We fit three models on the same training data. The lone **decision tree** is the weak baseline. The **random forest** grows 200 trees on bootstrapped samples and averages their votes. **Gradient boosting** grows trees in sequence, each one correcting the previous trees' mistakes.

In [ ]:
tree = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
rf = RandomForestClassifier(n_estimators=200, random_state=0).fit(Xtr, ytr)
gb = GradientBoostingClassifier(random_state=0).fit(Xtr, ytr)

### Step 3 — Score each model on the held-out test set

We measure accuracy on the test set none of the models trained on. The single tree usually trails, while the two ensembles score higher — combining many trees smooths out the noise that fools any one tree.

In [ ]:
tree_acc = tree.score(Xte, yte)
rf_acc = rf.score(Xte, yte)
gb_acc = gb.score(Xte, yte)

print("single tree   :", round(tree_acc, 3))
print("random forest :", round(rf_acc, 3))
print("boosting      :", round(gb_acc, 3))

## Visualize the data & results

_On breast-cancer data, does combining many trees beat a single decision tree?_

### Step 1 — Load real breast-cancer data and split it

Now repeat the comparison on real tumor scans. Each row is one patient's measurements, labeled malignant or benign. We again hold out 25% as a test set so the bar chart reflects true generalization, not memorized training rows.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split

bc = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(bc.data, bc.target, test_size=0.25, random_state=0)

### Step 2 — Train all three models and record test accuracy

We fit the same three model types on the real training data and immediately score each on the test set, storing the three accuracies so we can chart them side by side.

In [ ]:
tree = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr).score(Xte, yte)
rf = RandomForestClassifier(n_estimators=200, random_state=0).fit(Xtr, ytr).score(Xte, yte)
gb = GradientBoostingClassifier(random_state=0).fit(Xtr, ytr).score(Xte, yte)

### Step 3 — Compare them in a bar chart

A bar chart makes the gap obvious: the single tree's bar sits lower than the two ensemble bars. On real data, as on synthetic data, letting many trees vote beats trusting one.

In [ ]:
plt.bar(["single tree", "random forest", "boosting"], [tree, rf, gb],
        color=["#ff7b72", "#7ee787", "#ffb454"])
plt.ylabel("test accuracy")
plt.title("Single tree vs ensembles")
plt.show()